In [15]:
import os
import numpy as np
import joblib

from PIL import Image
from sklearn.svm import SVC
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

In [16]:
data_dir = "dataset"  # sửa nếu cần

X = []
y = []
label_map = {}
label_id = 0

if not os.path.exists(data_dir):
    raise ValueError("❌ Không tìm thấy thư mục dataset!")

for person in os.listdir(data_dir):
    person_path = os.path.join(data_dir, person)

    if not os.path.isdir(person_path):
        continue

    label_map[label_id] = person

    for img_name in os.listdir(person_path):
        img_path = os.path.join(person_path, img_name)

        try:
            img = Image.open(img_path).convert("L")
            img = img.resize((64, 64))

            X.append(np.array(img).flatten())
            y.append(label_id)

        except:
            continue

    label_id += 1

X = np.array(X)
y = np.array(y)

print("✅ Dataset shape:", X.shape)
print("✅ Labels:", label_map)

✅ Dataset shape: (31, 4096)
✅ Labels: {0: 'messi', 1: 'neymar', 2: 'ronaldo'}


In [17]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

In [18]:
# Auto chọn số chiều hợp lệ
n_samples = X_scaled.shape[0]
n_components = min(50, n_samples - 1)

pca = PCA(n_components=n_components)
X_pca = pca.fit_transform(X_scaled)

print("Số ảnh:", n_samples)
print("Số chiều PCA:", n_components)
print("Shape sau PCA:", X_pca.shape)

Số ảnh: 31
Số chiều PCA: 30
Shape sau PCA: (31, 30)


In [19]:
svm = SVC(kernel='rbf', probability=True)
svm.fit(X_pca, y)

print("✅ Train xong!")

✅ Train xong!


In [20]:
os.makedirs("models", exist_ok=True)

joblib.dump(svm, "models/svm_face.pkl")
joblib.dump(pca, "models/pca.pkl")
joblib.dump(scaler, "models/scaler.pkl")
joblib.dump(label_map, "models/labels.pkl")

print("✅ Model saved!")

✅ Model saved!


In [21]:
svm = joblib.load("models/svm_face.pkl")
pca = joblib.load("models/pca.pkl")
scaler = joblib.load("models/scaler.pkl")
label_map = joblib.load("models/labels.pkl")

print("✅ Model loaded!")

✅ Model loaded!
